In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import json
from sklearn.decomposition import PCA
from PIL import Image
import yaml

import torch
import torch.nn.functional as F
import torch.nn as nn
from torchvision.transforms import v2
import torchvision.transforms as transforms
from torchvision.models import vit_l_16, ViT_L_16_Weights
from torchvision.io import decode_image

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.base_cam import BaseCAM

from scripts.model import Custom_ResNet50, Custom_DenseNet121, ModelWrapper
from scripts.preprocessing import get_hilbert_index, hilbert_ravel, pipeline
import scripts.gradcam

# View Grad-CAM Mask Similarities Between Conditions

Although

In [2]:
# Load variables

##### Import common.yaml
with open("config/common.yaml", "r") as common_params:
    common = yaml.safe_load(common_params)
    # unpack params
    conditions = common["conditions"]
    class_cols = common["class_cols"]
    n_classes = common["n_classes"]

### Load wrapper_kwargs
with open("config/wrapper_kwargs.yaml", "r") as params:
    wrapper_kwargs = yaml.safe_load(params)

In [3]:
model_types = ["ResNet50", "ResNet50", "ResNet50", "ResNet50", 
               "DenseNet121", "DenseNet121", "DenseNet121", "DenseNet121"]
dims = [224, 224, 224, 224,
        384, 384, 384, 384]
enhanceds = [False, True, False, True,
             False, True, False, True]

model_types = ["ResNet50", "ResNet50", "DenseNet121", "DenseNet121"]
dims = [224, 224, 224, 224]
enhanceds = [False, True, False, True]


In [4]:
### Number of X-rays to loop through for each model
pca_sample_size = 100

### Number of PCA components
n_pca_components = 10

### Loop through the models

In [5]:
output_dict = {}
for i,(model_type, dim, enhanced) in enumerate(zip(model_types, dims, enhanceds)):
    t0 = time.time()
    label = str(model_type) + "_" + str(dim) + "_" + str("ENHANCED" if enhanced==True else "RAW")
    print(f"Starting analysis on {label}")
    
    ##### Assemble the variables
    wrapper_kwargs_i = wrapper_kwargs[f"wrapper_kwargs{i}"]
    wrapper_kwargs_i["pca_sample_size"] = pca_sample_size
    wrapper_kwargs_i["n_pca_components"] = n_pca_components
    
    #####
    output_dict[label] = scripts.gradcam.gradcam_pca_analysis(**wrapper_kwargs_i)
    print(f"\tComplete ---> Time elapsed: {round(time.time() - t0, 2)} s")



Starting analysis on ResNet50_224_RAW
	Complete ---> Time elapsed: 31.59 s
Starting analysis on ResNet50_224_ENHANCED
	Complete ---> Time elapsed: 28.96 s
Starting analysis on DenseNet121_224_RAW
	Complete ---> Time elapsed: 81.45 s
Starting analysis on DenseNet121_224_ENHANCED
	Complete ---> Time elapsed: 80.23 s


In [8]:
output_dict

{'ResNet50_224_RAW': ({'avg_dot': array([[0.92840509, 0.92825214, 0.91887575, ..., 0.        , 0.        ,
           0.        ],
          [0.92825214, 0.93441982, 0.91881437, ..., 0.        , 0.        ,
           0.        ],
          [0.91887575, 0.91881437, 1.85330651, ..., 0.        , 0.        ,
           0.        ],
          ...,
          [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
           0.        ],
          [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
           0.        ],
          [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
           0.        ]]),
   'avg_L2': array([[0.00000000e+00, 7.94809201e-05, 9.66754336e-03, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [7.94809201e-05, 0.00000000e+00, 9.74511994e-03, ...,
           0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
          [9.66754336e-03, 9.74511994e-03, 0.00000000e+00, ...,
           0.00000000e+00, 0.0

In [6]:
# Save analysis
results_filename = "results/gradcam/PCA_analysis_dicts.json"

with open(results_filename, "w") as fh:
    fh.write(str(output_dict))

In [10]:
# Open analysis for viewing
with open(results_filename, "r") as fh:
    results_dict = eval(fh.read().replace("array", "np.array"))

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (7,) + inhomogeneous part.

In [ ]:
print("PCA_analysis_dicts Structure\n")
for key1,values1 in results_dict.items():
    print(key1)
    for key2,values2 in values1.items(): 
        print(f"\t{key2}")

In [ ]:
results_dict

In [ ]:
### Plot average explained variance
avg_explained_variance = {}
for mod1,values1 in results_dict.items():
    avg_explained_variance[mod1] = results_dict[mod1]["avg_explained_variance"]

avg_explained_variance